<a href="https://colab.research.google.com/github/anirban4747/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anirban4747/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
import duckdb
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.sql(f"CREATE SECRET hf_token (TYPE huggingface, TOKEN '{hf_token}')")

# Sanity check: this should list files without erroring if access + token are both correct
con.sql("""
SELECT * FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
LIMIT 5
""").show()

┌─────────────┬─────────────────────────┬──────────────────────────┬────────────────┬────────────────┬────────────────────┬────────────────────┬─────────────────┬────────────┬──────────────────┬───────────────────┬───────────────┬──────────────┬───────────┬──────────────────────┬──────────────────────────┬──────────────────┬─────────────────┬───────────────────┬─────────────────┬───────────────┬─────────────┬────────────┬───────────────┬───────────┬────────────┬───────────┬─────────┬──────────┬───────────────┬─────────┐
│ report_date │     client_hash_id      │     content_hash_id      │ client_has_gsc │ client_has_ga4 │ gsc_data_available │ ga4_data_available │ gsc_impressions │ gsc_clicks │ gsc_sum_position │ gsc_avg_position  │ ga4_pageviews │ ga4_sessions │ ga4_users │ ga4_engaged_sessions │ ga4_total_engagement_sec │ sessions_organic │ sessions_direct │ sessions_referral │ sessions_social │ sessions_paid │ sessions_ai │ ai_chatgpt │ ai_perplexity │ ai_gemini │ ai_copilot │ ai_clau

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [ ]:
print("""
DATA CONTRACT — Lane 2: Refresh / Content Opportunity Scoring

1. Unit of analysis: one row = one content page, on one day
   (grain: report_date x client_hash_id x content_hash_id).

2. Table(s): fact_content_daily_performance, month=2026-03 partition
   (mid-panel month, per the warning — never the _sample table for label logic,
   since _sample IS June 2026, the sealed final month).
   Joined to dim_content (content_hash_id) for content metadata where needed.

3. Time window: report_date between 2026-03-01 and 2026-03-31.

4. What I'd predict/rank: a proxy for "worth reviewing" built from within-window
   signal movement (e.g. comparing early-month vs late-month impressions/position
   for the same page) — NOT the starter CSV's trend_direction, which doesn't exist
   at this daily grain and is itself derived from a proxy. Full label definition
   is deferred to the modeling weeks; this week's contract only commits to the
   grain and window the label will be built from.

5. Deliberately excluded: raw query/url/title fields (not shipped anyway), and
   any product decision flags (health_score, priority_score, action_type) —
   none are shipped in this data by design, so there's nothing to strip, but I
   confirm I will not attempt to reconstruct one and feed it back in as a feature.
""")


DATA CONTRACT — Lane 2: Refresh / Content Opportunity Scoring

1. Unit of analysis: one row = one content page, on one day
   (grain: report_date x client_hash_id x content_hash_id).

2. Table(s): fact_content_daily_performance, month=2026-03 partition
   (mid-panel month, per the warning — never the _sample table for label logic,
   since _sample IS June 2026, the sealed final month).
   Joined to dim_content (content_hash_id) for content metadata where needed.

3. Time window: report_date between 2026-03-01 and 2026-03-31.

4. What I'd predict/rank: a proxy for "worth reviewing" built from within-window
   signal movement (e.g. comparing early-month vs late-month impressions/position
   for the same page) — NOT the starter CSV's trend_direction, which doesn't exist
   at this daily grain and is itself derived from a proxy. Full label definition
   is deferred to the modeling weeks; this week's contract only commits to the
   grain and window the label will be built from.

5. Deliber

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [ ]:
print("""
FIELD BUCKETS

Context (never modeled on):
  - client_hash_id, content_hash_id, report_date  -> joining/grouping/splitting only

Feature (knowable at the decision moment):
  - gsc_avg_position, gsc_impressions, gsc_clicks  -> observed search signal
  - ga4_sessions (only where ga4_data_available IS TRUE)
  - scroll/engagement columns where available

Label / proxy (never a feature):
  - whatever within-window or future-window decline signal I define for
    modeling weeks -- placeholder for now, will be built from the same
    observed columns above, over a LATER window than the features

Excluded:
  - any product-decision flag (not shipped, and would be excluded even if it were)
  - raw text/query/url fields (not shipped)
""")


FIELD BUCKETS

Context (never modeled on):
  - client_hash_id, content_hash_id, report_date  -> joining/grouping/splitting only

Feature (knowable at the decision moment):
  - gsc_avg_position, gsc_impressions, gsc_clicks  -> observed search signal
  - ga4_sessions (only where ga4_data_available IS TRUE)
  - scroll/engagement columns where available

Label / proxy (never a feature):
  - whatever within-window or future-window decline signal I define for
    modeling weeks -- placeholder for now, will be built from the same
    observed columns above, over a LATER window than the features

Excluded:
  - any product-decision flag (not shipped, and would be excluded even if it were)
  - raw text/query/url fields (not shipped)



## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
# ============================================================
# QUERY 1 — Grain check: does one row really mean what I claimed?
# ============================================================
print("=== Query 1: Grain check ===")
con.sql("""
SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
GROUP BY report_date, client_hash_id, content_hash_id
HAVING COUNT(*) > 1
LIMIT 5
""").show()
print("Empty result = grain holds: one row per page-day, confirmed.\n")


# ============================================================
# QUERY 2 — Row count + date span for this slice
# ============================================================
print("=== Query 2: Row count + date span ===")
con.sql("""
SELECT COUNT(*) AS row_count, MIN(report_date) AS min_date, MAX(report_date) AS max_date
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
""").show()
print("Confirms this slice's actual size and whether it spans the full month.\n")


# ============================================================
# QUERY 3 — Availability, filtered with IS TRUE
# ============================================================
print("=== Query 3: Availability check ===")
con.sql("""
SELECT
  COUNT(*) AS total_rows,
  SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS rows_with_ga4,
  ROUND(100.0 * SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_with_ga4
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
""").show()
print("Confirms only a fraction of rows have real GA4 data — matches the")
print("panel warning that GA4 columns are zero-filled (not truly zero) before ga4_data_start.\n")


# ============================================================
# FIVE FEATURES — build the frame
# ============================================================
print("=== Building five-feature frame ===")
features_df = con.sql("""
SELECT
  content_hash_id,
  client_hash_id,
  report_date,
  gsc_impressions,
  gsc_avg_position,
  gsc_clicks,
  CASE WHEN gsc_impressions > 0 THEN gsc_clicks * 1.0 / gsc_impressions ELSE NULL END AS ctr,
  ga4_sessions
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
WHERE ga4_data_available IS TRUE
LIMIT 5000
""").df()

display(features_df.head())

print("""
FIVE FEATURES — each knowable at the decision moment because:

1. gsc_impressions — a completed daily search-console measurement, logged
   after the day ends, before any refresh decision is made.

2. gsc_avg_position — an observed ranking outcome for that day, not
   dependent on any future action.

3. gsc_clicks — same as impressions: a closed daily measurement.

4. ctr (clicks/impressions) — purely derived from two features already
   known at the same moment; no future information involved.

5. ga4_sessions — knowable, but ONLY for rows where ga4_data_available IS
   TRUE; using it elsewhere would silently inject a "client has GA4
   tracking" signal disguised as a traffic signal.
""")


# ============================================================
# THE TRAP — add a label-derived column on purpose, then remove it
# ============================================================
print("=== The leakage trap ===")
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score

demo = features_df.dropna(subset=["gsc_avg_position", "ctr"]).copy()
demo["demo_label"] = (demo["gsc_avg_position"] > demo["gsc_avg_position"].median()).astype(int)

# Honest features only
X_honest = demo[["gsc_impressions", "ctr", "gsc_clicks"]].fillna(0)
y = demo["demo_label"]

clf = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_honest, y)
honest_auc = roc_auc_score(y, clf.predict_proba(X_honest)[:, 1])
print(f"Honest AUC (no leak): {honest_auc:.3f}")

# Spring the trap: add a column derived directly from the label
demo["leaky_col"] = demo["gsc_avg_position"]  # this IS what demo_label was thresholded on
X_leaky = demo[["gsc_impressions", "ctr", "gsc_clicks", "leaky_col"]].fillna(0)

clf_leaky = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_leaky, y)
leaky_auc = roc_auc_score(y, clf_leaky.predict_proba(X_leaky)[:, 1])
print(f"Leaky AUC (with gsc_avg_position, the label's own source): {leaky_auc:.3f}  <- jumps toward 1.0")

print("\nDeleting leaky_col and keeping the honest number.")
del demo["leaky_col"]
print(f"Final honest AUC kept for this contract: {honest_auc:.3f}")

=== Query 1: Grain check ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬────────────────┬─────────────────┬───────┐
│ report_date │ client_hash_id │ content_hash_id │   c   │
│    date     │    varchar     │     varchar     │ int64 │
├─────────────┴────────────────┴─────────────────┴───────┤
│                         0 rows                         │
└────────────────────────────────────────────────────────┘

Empty result = grain holds: one row per page-day, confirmed.

=== Query 2: Row count + date span ===
┌───────────┬────────────┬────────────┐
│ row_count │  min_date  │  max_date  │
│   int64   │    date    │    date    │
├───────────┼────────────┼────────────┤
│   9841378 │ 2026-03-01 │ 2026-03-31 │
└───────────┴────────────┴────────────┘

Confirms this slice's actual size and whether it spans the full month.

=== Query 3: Availability check ===
┌────────────┬───────────────┬──────────────┐
│ total_rows │ rows_with_ga4 │ pct_with_ga4 │
│   int64    │    int128     │    double    │
├────────────┼───────────────┼──────────────┤
│    984137

,content_hash_id,client_hash_id,report_date,gsc_impressions,gsc_avg_position,gsc_clicks,ctr,ga4_sessions
0,content_09be8cc7fcb222af,client_65de48885f4ef01b,2026-03-01,0,NaN,0,NaN,1
1,content_851afac9fe13612e,client_65de48885f4ef01b,2026-03-01,0,NaN,0,NaN,1
2,content_cee6c6fc8c51af14,client_65de48885f4ef01b,2026-03-01,0,NaN,0,NaN,1
3,content_5e120e972f11f833,client_65de48885f4ef01b,2026-03-01,0,NaN,0,NaN,1
4,content_16a7291bb6ecaebe,client_65de48885f4ef01b,2026-03-01,0,NaN,0,NaN,1



FIVE FEATURES — each knowable at the decision moment because:

1. gsc_impressions — a completed daily search-console measurement, logged
   after the day ends, before any refresh decision is made.

2. gsc_avg_position — an observed ranking outcome for that day, not
   dependent on any future action.

3. gsc_clicks — same as impressions: a closed daily measurement.

4. ctr (clicks/impressions) — purely derived from two features already
   known at the same moment; no future information involved.

5. ga4_sessions — knowable, but ONLY for rows where ga4_data_available IS
   TRUE; using it elsewhere would silently inject a "client has GA4
   tracking" signal disguised as a traffic signal.

=== The leakage trap ===
Honest AUC (no leak): 0.728
Leaky AUC (with gsc_avg_position, the label's own source): 1.000  <- jumps toward 1.0

Deleting leaky_col and keeping the honest number.
Final honest AUC kept for this contract: 0.728


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [ ]:
print("""
LIMITATION: This slice (month=2026-03) is a single mid-panel month for
~70 clients with wildly uneven history depth (per dim_clients.gsc_data_start).
Some clients may have little or no usable history by this month, meaning
this one-month slice is not representative of the full client base — any
pattern found here should be re-checked against other months before being
treated as a general finding.
""")


LIMITATION: This slice (month=2026-03) is a single mid-panel month for
~70 clients with wildly uneven history depth (per dim_clients.gsc_data_start).
Some clients may have little or no usable history by this month, meaning
this one-month slice is not representative of the full client base — any
pattern found here should be re-checked against other months before being
treated as a general finding.



## Self-check

Before you submit, confirm each line honestly:

- [x ] Every section above is filled — markdown thinking AND the code that backs it
- [ x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ x] No client names, URLs, or private queries anywhere
- [ x] My claims use careful words: observed, measured, directional, decision-support
- [x ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.